# A2 独立评估

实际生成的 1000 对原始／复制区域 Pearson；绘制小提琴和均值柱状图，另存自然区域 Spearman。

使用 simscDNAm 环境的 Jupyter，修改 `evaluate/config.yaml` 后从上到下运行。本入口只执行本实验，结果写入配置输出目录的 `A2/`，图写入其 `figures/`。

In [ ]:
from pathlib import Path
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
CONFIG_PATH = PROJECT / "evaluate/config.yaml"
EXPERIMENT = "A2"

## 读取配置与指标

相对路径以配置文件所在目录为准；各实验使用同一随机种子和同一批抽样细胞。

In [ ]:
from contextlib import ExitStack
import importlib.metadata
import json
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
import yaml

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
sys.path.insert(0, str(PROJECT))
from data.evaluation import (dense_block, read_labels, region_pairs, validate_real_duplicates,
    natural_reference_statistics, select_features, imputed_features, validate_region_axis, sample_rows)
if Path(sys.prefix).name.lower() != "simscdnam":
    raise RuntimeError(f"请切换到 simscDNAm 的 Jupyter 内核，当前为 {sys.executable}")
CONFIG_PATH = CONFIG_PATH.expanduser().resolve()
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(config, dict):
    raise ValueError("配置必须是 YAML 映射")
required = {"real_h5ad", "synthetic_h5ad", "output_dir"}
if not required.issubset(config) or not isinstance(config["synthetic_h5ad"], dict) or not config["synthetic_h5ad"]:
    raise ValueError("请填写 real_h5ad、非空 synthetic_h5ad 方法映射和 output_dir")
defaults = {"reference_h5ad": None, "region_pairs_csv": None, "label_key": "Subclass",
    "duplicate_count": 1000, "seed": 42, "evaluation_cells": None, "a3_top_regions": 50000,
    "pca_components": 50, "milisi_neighbors": 30, "umap_seed": 2022, "spearman_top_regions": 2000,
    "a1_regions_csv": None, "genome_assembly": None, "read_in_memory": False}
config = {**defaults, **config}
if not isinstance(config["read_in_memory"], bool):
    raise ValueError("read_in_memory必须为布尔值")
if config["label_key"] != "Subclass":
    raise ValueError("本流程统一使用真实类型 Subclass，请将真实标签字段改名为 Subclass")
if config["duplicate_count"] != 1000:
    raise ValueError("本流程固定最后 1000 个区间为复制区间")
for key in ("a3_top_regions", "pca_components", "milisi_neighbors", "spearman_top_regions"):
    if not isinstance(config[key], int) or isinstance(config[key], bool) or config[key] < 2:
        raise ValueError(f"{key} 必须为不小于 2 的整数")
if config["evaluation_cells"] is not None and (not isinstance(config["evaluation_cells"], int)
        or isinstance(config["evaluation_cells"], bool) or config["evaluation_cells"] < 3):
    raise ValueError("evaluation_cells 必须为 null 或不小于 3 的整数")
if any(not isinstance(name, str) or not name.strip() or name in {"真实数据", "训练参照", "CVAEKMeansAligned"}
       for name in config["synthetic_h5ad"]):
    raise ValueError("方法名须为非空字符串，且不能使用保留名称或暂不参与的 CVAEKMeansAligned")

def resolve_path(value):
    path = Path(value).expanduser()
    return (CONFIG_PATH.parent / path).resolve() if not path.is_absolute() else path.resolve()

plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans", "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Jupyter 内核：", sys.executable)
print("配置文件：", CONFIG_PATH)
from evaluate.A2.metrics import paired_pearson, spearman_vector


## 本实验绘图

图中文字使用英文，缺失值和超出 0–1 的值按原协议处理。

In [ ]:
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


PLOT_COLORS = {"真实数据": "#A6A6A6", "Real": "#A6A6A6",
    "simscDNAm": "#3B73B9", "CVAE_training_mask": "#3B73B9", "FittedMarginal": "#269D91",
    "simATAC": "#A487C9", "simCAS": "#D95F02", "STADiffuser": "#B44C76"}


def plot_name(name):
    return {"真实数据": "Real", "CVAE_training_mask": "CVAE (training mask)"}.get(name, name)


def plot_color(name):
    return PLOT_COLORS.get(name, "#3B73B9")


def style_axis(axis):
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_linewidth(0.7)
    axis.grid(axis="y", color="#E8E8E8", linewidth=0.5)
    axis.set_axisbelow(True)


def save_figure(figure, output, stem):
    with plt.rc_context({"pdf.fonttype": 42}):
        for extension in ("png", "pdf"):
            figure.savefig(output / "figures" / f"{stem}.{extension}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def draw_correlation_violin(axis, vector, position, color, seed):
    finite = np.asarray(vector)[np.isfinite(vector)]
    if not finite.size:
        axis.text(position, 0, "No valid pairs", rotation=90, ha="center", fontsize=8)
        return
    median = np.median(finite)
    values = np.random.default_rng(seed).choice(finite, 200000, replace=False) if finite.size > 200000 else finite
    if values.size > 1 and np.ptp(values) > 1e-12:
        violin = axis.violinplot(values, positions=[position], widths=0.8, showextrema=False, points=200)
        for body in violin["bodies"]:
            body.set(facecolor=color, edgecolor="none", alpha=0.72)
        axis.hlines(median, position - 0.13, position + 0.13,
                    color="white" if color == "#D95F02" else "#E8793E", linewidth=1)
    else:
        axis.hlines(median, position - 0.25, position + 0.25, color=color, linewidth=1.4)


def plot_duplicate_pearson(correlations, summary, output):
    """小提琴展示有效对分布；柱高直接读取正式汇总，避免重画改变得分。"""
    methods = summary.loc[~summary["方法"].eq("真实数据"), "方法"].tolist()
    table = summary.set_index("方法").loc[methods]
    pair_count = int(summary["配对数"].max())
    figure, axes = plt.subplots(1, 2, figsize=(12.8, 4.7))
    figure.subplots_adjust(left=0.06, right=0.995, bottom=0.26, top=0.84, wspace=0.12)
    figure.suptitle(f"Dependency preservation after appending {pair_count:,} duplicated regions", fontsize=12, weight="bold", y=0.98)
    for position, method in enumerate(["真实数据", *methods], 1):
        vector = correlations.loc[correlations["方法"].eq(method), "Pearson"].to_numpy()
        draw_correlation_violin(axes[0], vector, position, plot_color(method), 42)
    axes[0].axhline(1, color="#888888", linestyle="--", linewidth=0.8, zorder=0)
    axes[0].set(title="Original vs duplicated regions", ylabel="Pearson correlation", ylim=(-1.05, 1.05),
                xticks=np.arange(1, len(methods) + 2), xticklabels=[plot_name(name) for name in ["真实数据", *methods]])
    means = table["Pearson均值"].to_numpy(dtype=float)
    axes[1].bar(np.arange(len(methods)), means, color=[plot_color(name) for name in methods], width=0.8)
    finite = means[np.isfinite(means)]
    lower = min(-0.05, float(finite.min()) * 1.25) if len(finite) else -0.05
    axes[1].set(title="Mean Pearson across valid region pairs", ylabel="Mean Pearson (higher is better)",
                ylim=(lower, 1.12), xticks=np.arange(len(methods)), xticklabels=[plot_name(name) for name in methods])
    for position, value in enumerate(means):
        label = f"{value:.4f}" if np.isfinite(value) else "NA"
        y = value + (0.02 if value >= 0 else -0.02) if np.isfinite(value) else 0.02
        axes[1].text(position, y, label, ha="center", va="bottom" if not np.isfinite(value) or value >= 0 else "top", fontsize=8)
    for axis in axes:
        style_axis(axis)
        axis.title.set_fontsize(10)
        axis.tick_params(axis="x", labelrotation=30, labelsize=8)
        for label in axis.get_xticklabels():
            label.set_ha("right")
    counts = "; ".join(f"{plot_name(name)} {int(table.loc[name, '有效配对数']):,}/{int(table.loc[name, '配对数']):,}" for name in methods)
    figure.text(0.06, 0.015, "Valid pairs: " + counts, fontsize=7, color="#555555")
    save_figure(figure, output, "A2_duplicate_Pearson")

## 执行与导出

校验完整区域轴及真实复制列；独立训练参照与真实留出细胞不得重叠。失败写入运行记录，读取对象在结束时关闭。

In [ ]:
def evaluate_experiment(config):
    methods = list(config["synthetic_h5ad"])
    paths = {"真实数据": resolve_path(config["real_h5ad"]),
             **{name: resolve_path(value) for name, value in config["synthetic_h5ad"].items()}}
    reference_path = resolve_path(config["reference_h5ad"]) if config["reference_h5ad"] else paths["真实数据"]
    pair_path = resolve_path(config["region_pairs_csv"]) if config["region_pairs_csv"] else None
    region_path = resolve_path(config["a1_regions_csv"]) if config.get("a1_regions_csv") else None
    output = resolve_path(config["output_dir"]) / EXPERIMENT
    if sys.platform != "win32":
        output.relative_to(Path("/home/sccasimp/data/xb/outputs/simscDNAm").resolve())
    all_paths = set(paths.values()) | {reference_path}
    if pair_path:
        all_paths.add(pair_path)
    if EXPERIMENT == "A1" and region_path:
        all_paths.add(region_path)
    if any(path.is_relative_to(output) for path in all_paths):
        raise ValueError("输出目录不能包含输入文件，以免覆盖输入；请选择独立的评估输出目录")
    for path in all_paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少输入文件：{path}")
    if len(set(paths.values())) != len(paths):
        raise ValueError("真实数据和各方法合成数据必须指向不同文件")
    records = [{"路径": str(path), "字节数": path.stat().st_size, "修改时间纳秒": path.stat().st_mtime_ns}
               for path in sorted(all_paths)]
    output.mkdir(parents=True, exist_ok=True)
    (output / "figures").mkdir(exist_ok=True)
    report = {"状态": "运行中", "标签字段": "Subclass", "最后复制区间数": 1000, "方法": methods,
              "参照模式": "独立训练真实参照" if reference_path != paths["真实数据"] else "评估真实数据拟合参照，非独立留出评估",
              "配置": config, "配置路径": str(CONFIG_PATH), "输入文件": records, "内核": sys.executable,
              "依赖版本": {name: importlib.metadata.version(name) for name in
                  ["numpy", "pandas", "scipy", "anndata", "scikit-learn", "umap-learn"]}}
    def write_report():
        (output / "run_summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    write_report()
    try:
        with ExitStack() as stack, threadpool_limits(limits=8):
            datasets = {}
            for name, path in paths.items():
                item = ad.read_h5ad(path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets[name] = item
            real = datasets["真实数据"]
            if reference_path == paths["真实数据"]:
                datasets["训练参照"] = real
            else:
                item = ad.read_h5ad(reference_path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets["训练参照"] = item
                if len(real.obs_names.intersection(item.obs_names)):
                    raise ValueError("独立训练参照与评估真实细胞 ID 重叠，请提供分离后的数据")
            for name, item in datasets.items():
                validate_region_axis(real, item)
                if not item.obs_names.is_unique:
                    raise ValueError(f"{name} 的细胞 ID 不唯一")
            pairs = region_pairs(real, pair_path=pair_path)
            validate_real_duplicates(real, pairs)
            if datasets["训练参照"] is not real:
                validate_real_duplicates(datasets["训练参照"], pairs)
            natural_count = real.n_vars - 1000
            count = real.n_obs if config["evaluation_cells"] is None else config["evaluation_cells"]
            rows = {name: sample_rows(item, count, config["seed"]) for name, item in datasets.items() if name != "训练参照"}
            rows["训练参照"] = np.arange(datasets["训练参照"].n_obs)
            labels = {name: read_labels(item, rows[name]) for name, item in datasets.items()}
            if EXPERIMENT == "A3" and 2 * count <= config["milisi_neighbors"]:
                raise ValueError("合并细胞数必须大于 miLISI 近邻数")
            pairs.to_csv(output / "region_pairs.csv", index=False)
            pd.concat([pd.DataFrame({"数据集": name, "原行索引": selected,
                "细胞ID": datasets[name].obs_names[selected], "细胞类型": labels[name]})
                for name, selected in rows.items()], ignore_index=True).to_csv(output / "selected_cells.csv", index=False)
            report.update({"评估细胞数": count, "真实参照细胞数": len(rows["训练参照"]), "原始区间数": natural_count,
                           "总区间数": real.n_vars, "A1_A3排除末尾复制列": True})
            write_report()
            print(f"输入校验通过：原始区间 {natural_count}，复制区间 1000，每种方法评估 {count} 个细胞。", flush=True)

            print("开始 A2：读取实际复制列，计算 1000 对 Pearson。", flush=True)
            pair_tables = []
            for name in ["真实数据", *methods]:
                original = dense_block(datasets[name], rows[name], pairs["原region索引"].to_numpy())
                copied = dense_block(datasets[name], rows[name], pairs["复制region索引"].to_numpy())
                frame = pd.concat([pairs, paired_pearson(original, copied)], axis=1)
                frame.insert(0, "方法", name)
                pair_tables.append(frame)
            correlations = pd.concat(pair_tables, ignore_index=True)
            a2 = correlations.groupby("方法", sort=False).agg(配对数=("Pearson", "size"),
                有效配对数=("Pearson", "count"), Pearson均值=("Pearson", "mean"), Pearson中位数=("Pearson", "median")).reset_index()
            a2["无效配对数"] = a2["配对数"] - a2["有效配对数"]
            correlations.to_csv(output / "A2_duplicate_correlations.csv", index=False)
            a2.to_csv(output / "A2_pearson_summary.csv", index=False)
            plot_duplicate_pearson(correlations, a2, output)

            print("仅用真实参照的原始区间拟合中位数、按 Subclass 的 ANOVA。", flush=True)
            fill, scores = natural_reference_statistics(datasets["训练参照"], rows["训练参照"], natural_count)
            selected = select_features(scores, config["spearman_top_regions"])
            pd.DataFrame({"原region索引": selected, "原region": real.var_names[selected], "ANOVA": scores[selected]}).to_csv(
                output / "A2_natural_selected_regions.csv", index=False)
            spearman_rows = []
            for index, name in enumerate(["真实数据", *methods]):
                vector = spearman_vector(imputed_features(datasets[name], rows[name], selected, fill))
                filename = f"A2_Spearman_{index:02d}.npy"
                np.save(output / filename, vector)
                finite = vector[np.isfinite(vector)]
                spearman_rows.append({"方法": name, "入选原始区域数": len(selected), "区域对数": len(vector),
                    "有效区域对数": len(finite), "Spearman均值": finite.mean() if len(finite) else np.nan,
                    "Spearman中位数": np.median(finite) if len(finite) else np.nan, "向量文件": filename})
            pd.DataFrame(spearman_rows).to_csv(output / "A2_natural_Spearman_summary.csv", index=False)

            summary = a2
            diagnostics = []
            for name in ["真实数据", *methods]:
                for domain, start, stop in [("原始区间", 0, natural_count), ("复制区间", natural_count, real.n_vars)]:
                    valid, outside = 0, 0
                    for left in range(start, stop, 512):
                        block = dense_block(datasets[name], rows[name], slice(left, min(left + 512, stop)))
                        finite = np.isfinite(block)
                        valid += int(finite.sum())
                        outside += int((finite & ((block < 0) | (block > 1))).sum())
                    diagnostics.append({"数据集": name, "区间范围": domain, "有限观测数": valid,
                        "缺失比例": 1 - valid / (len(rows[name]) * (stop - start)),
                        "超出0到1比例": outside / valid if valid else np.nan})
            pd.DataFrame(diagnostics).to_csv(output / "input_diagnostics.csv", index=False)
            for record in records:
                path = Path(record["路径"])
                if path.stat().st_size != record["字节数"] or path.stat().st_mtime_ns != record["修改时间纳秒"]:
                    raise RuntimeError(f"评估期间输入文件发生变化：{path}")
            report.update({"状态": "完成", "输入文件大小及修改时间未变": True,
                           "输出目录": str(output), "输出文件": sorted(str(p.relative_to(output)) for p in output.rglob("*") if p.is_file())})
            write_report()
            display(summary)
            print(EXPERIMENT + " 评估完成，结果目录：", output)
            return summary
    except Exception as error:
        report.update({"状态": "失败", "错误": f"{type(error).__name__}: {error}"})
        write_report()
        raise


summary = evaluate_experiment(config)